# Willett Representation Manifolds

Export hidden states and phoneme predictions from trained Willett-style GRU/S5 decoders, then run first-pass low-dimensional analyses using broad phonetic categories rather than hard phoneme-window alignment.

The important contract is: extraction writes durable per-model artifacts under Drive, and analysis cells read those artifacts without touching checkpoints again.

In [ ]:
# Colab / Drive / repo bootstrap.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Not running in Colab or Drive already unavailable: {exc}')

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/utah-ssl') if Path('/content').exists() else Path.cwd()
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'

if str(REPO_DIR).startswith('/content'):
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(REPO_DIR), check=False)

os.chdir(REPO_DIR)
PACKAGE_ROOT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'
os.environ['PYTHONPATH'] = f"{PACKAGE_ROOT}:{os.environ.get('PYTHONPATH', '')}"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

# pyarrow is optional; without it the exporter still writes CSV tables.
try:
    import sklearn, seaborn, pyarrow  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'pyarrow'], check=False)

print('REPO_DIR:', REPO_DIR)
print('PACKAGE_ROOT:', PACKAGE_ROOT)

In [ ]:
from pathlib import Path

# Drive/local roots. Edit these in one place if your Drive layout differs.
DRIVE_ROOT = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('/Users/home/My Drive')
UTAH_SSL_ROOT = DRIVE_ROOT / 'utah_ssl'
RAW_CACHE_ROOT = UTAH_SSL_ROOT / 'data' / 'cache_v1'
REPRESENTATION_ROOT = UTAH_SSL_ROOT / 'data' / 'representations' / 'willett_manifolds'

EXPORT_NAME = 'gru_s5_val_area6v_soft_phonetic_categories_v1'
EXPORT_ROOT = REPRESENTATION_ROOT / EXPORT_NAME

DATASET = 'brain2text24'
EXPORT_SPLIT = 'val'
MAX_EXAMPLES = None      # set to a small int, e.g. 16, for a smoke extraction
BATCH_SIZE = 32
SHARD_SIZE_TOKENS = 50_000
OVERWRITE_EXPORT = False
DEVICE = None           # None auto-detects cuda/mps/cpu
BIN_SIZE_MS = 20

MODEL_CANDIDATES = {
    'gru': [
        UTAH_SSL_ROOT / 'outputs' / 'willett_reconstruction' / 'willett_tx_only_area6v_colab' / 'checkpoint_best.pt',
    ],
    's5': [
        UTAH_SSL_ROOT / 'outputs' / 'ssl_experiments' / 'willett_s5_tx_sbp' / 'willett_s5_tx_sbp_seed7_60k' / 'checkpoint_best.pt',
        UTAH_SSL_ROOT / 'outputs' / 'willett_s5_reconstruction' / '*' / 'checkpoint_best.pt',
    ],
}

EXPORT_ROOT

In [ ]:
import glob
import json
from IPython.display import display
import pandas as pd


def resolve_candidates(candidates):
    rows = []
    for candidate in candidates:
        candidate = Path(candidate)
        matches = sorted(Path(path) for path in glob.glob(str(candidate))) if '*' in str(candidate) else [candidate]
        for path in matches:
            rows.append({'candidate': str(path), 'exists': path.exists()})
    existing = [Path(row['candidate']) for row in rows if row['exists']]
    return (existing[0] if existing else None), pd.DataFrame(rows)

resolved_checkpoints = {}
resolution_tables = []
for model_key, candidates in MODEL_CANDIDATES.items():
    resolved, table = resolve_candidates(candidates)
    table.insert(0, 'model_key', model_key)
    resolution_tables.append(table)
    if resolved is not None:
        resolved_checkpoints[model_key] = resolved

checkpoint_resolution = pd.concat(resolution_tables, ignore_index=True) if resolution_tables else pd.DataFrame()
display(checkpoint_resolution)
print('Resolved checkpoints:')
print(json.dumps({key: str(value) for key, value in resolved_checkpoints.items()}, indent=2))

if not resolved_checkpoints:
    raise FileNotFoundError('No checkpoints resolved. Edit MODEL_CANDIDATES above to point at trained GRU/S5 checkpoints.')

## Export Representations

This cell is the durable extraction step. It rebuilds each model from its checkpoint config, applies the same eval-time preprocessing as Willett validation, and writes hidden/logit shards plus token/example tables.

In [ ]:
from willett_reconstruction import RepresentationExportConfig, export_willett_representations

export_summaries = {}
for model_key, checkpoint_path in resolved_checkpoints.items():
    print(f'\n=== exporting {model_key}: {checkpoint_path} ===')
    summary = export_willett_representations(
        RepresentationExportConfig(
            checkpoint_path=checkpoint_path,
            export_root=EXPORT_ROOT,
            model_key=model_key,
            split=EXPORT_SPLIT,
            max_examples=MAX_EXAMPLES,
            batch_size=BATCH_SIZE,
            shard_size_tokens=SHARD_SIZE_TOKENS,
            bin_size_ms=BIN_SIZE_MS,
            overwrite=OVERWRITE_EXPORT,
            device=DEVICE,
            repo_dir=REPO_DIR,
            cache_root_override=RAW_CACHE_ROOT,
        )
    )
    export_summaries[model_key] = summary
    print(json.dumps({
        'model_key': model_key,
        'export_dir': str(EXPORT_ROOT / model_key),
        'example_count': summary.get('example_count'),
        'token_count': summary.get('token_count'),
        'hidden_dim': summary.get('hidden_dim'),
        'checkpoint_step': summary.get('checkpoint_step'),
    }, indent=2))

## Load Exported Artifacts

Everything below uses the saved artifacts, not the model checkpoints.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path


def load_model_export(export_root, model_key):
    model_dir = Path(export_root) / model_key
    metadata = json.loads((model_dir / 'metadata.json').read_text())
    tokens = pd.read_csv(model_dir / 'tokens.csv')
    examples = pd.read_csv(model_dir / 'examples.csv')
    shard_rows = json.loads((model_dir / 'shards.json').read_text())
    hidden_parts = []
    logits_parts = []
    for shard in shard_rows:
        arrays = np.load(model_dir / 'shards' / shard['shard'])
        hidden_parts.append(arrays['hidden'])
        logits_parts.append(arrays['logits'])
    hidden = np.concatenate(hidden_parts, axis=0) if hidden_parts else np.zeros((0, 0), dtype=np.float32)
    logits = np.concatenate(logits_parts, axis=0) if logits_parts else np.zeros((0, 0), dtype=np.float32)
    if hidden.shape[0] != tokens.shape[0]:
        raise ValueError(f'{model_key}: hidden rows {hidden.shape[0]} != token rows {tokens.shape[0]}')
    return {'metadata': metadata, 'tokens': tokens, 'examples': examples, 'hidden': hidden, 'logits': logits}

exports = {model_key: load_model_export(EXPORT_ROOT, model_key) for model_key in resolved_checkpoints}
summary_rows = []
for model_key, payload in exports.items():
    meta = payload['metadata']
    summary_rows.append({
        'model': model_key,
        'examples': meta['example_count'],
        'tokens': meta['token_count'],
        'hidden_dim': meta['hidden_dim'],
        'checkpoint_step': meta['checkpoint_step'],
        'patch_ms': meta['patch_size_ms'],
        'stride_ms': meta['patch_stride_ms'],
    })
display(pd.DataFrame(summary_rows))

## PCA Views

These are descriptive projections. Interpret color gradients more than apparent clusters.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


def fit_pca_view(hidden, max_points=60_000, seed=7):
    rng = np.random.default_rng(seed)
    n = hidden.shape[0]
    if n > max_points:
        idx = np.sort(rng.choice(n, size=max_points, replace=False))
    else:
        idx = np.arange(n)
    x = hidden[idx].astype(np.float32, copy=False)
    x = StandardScaler().fit_transform(x)
    pca = PCA(n_components=3, random_state=seed)
    pcs = pca.fit_transform(x)
    return idx, pcs, pca

pca_payloads = {}
for model_key, payload in exports.items():
    idx, pcs, pca = fit_pca_view(payload['hidden'])
    frame = payload['tokens'].iloc[idx].reset_index(drop=True).copy()
    frame['pc1'] = pcs[:, 0]
    frame['pc2'] = pcs[:, 1]
    frame['pc3'] = pcs[:, 2]
    pca_payloads[model_key] = {'frame': frame, 'pca': pca, 'indices': idx}
    print(model_key, 'explained variance:', np.round(pca.explained_variance_ratio_, 4))

color_columns = ['vowel_prob', 'consonant_prob', 'blank_prob', 'silence_prob', 'entropy_bits']
for model_key, p in pca_payloads.items():
    frame = p['frame']
    fig, axes = plt.subplots(1, len(color_columns), figsize=(4.2 * len(color_columns), 3.8), sharex=True, sharey=True)
    for ax, column in zip(axes, color_columns):
        sc = ax.scatter(frame['pc1'], frame['pc2'], c=frame[column], s=3, cmap='viridis', alpha=0.65, linewidths=0)
        ax.set_title(f'{model_key}: {column}')
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

## Category And Transition Diagnostics

In [ ]:
category_cols = [
    'blank_prob', 'silence_prob', 'vowel_prob', 'stop_prob', 'fricative_prob',
    'affricate_prob', 'nasal_prob', 'liquid_prob', 'glide_prob', 'consonant_prob',
]

for model_key, payload in exports.items():
    tokens = payload['tokens']
    print('\n', model_key)
    display(tokens[category_cols].describe().T[['mean', 'std', '25%', '50%', '75%']])
    transition_counts = tokens['transition_type'].value_counts().rename_axis('transition_type').reset_index(name='count')
    display(transition_counts.head(20))

    plot_frame = tokens[tokens['transition_type'].isin(['stable_or_first', 'vowel_to_consonant', 'consonant_to_vowel'])]
    if not plot_frame.empty:
        plt.figure(figsize=(7, 3.5))
        sns.boxplot(data=plot_frame, x='transition_type', y='entropy_bits')
        plt.title(f'{model_key}: entropy by broad transition type')
        plt.xticks(rotation=20, ha='right')
        plt.tight_layout()
        plt.show()

## High-Confidence Linear Probes

This checks whether broad category labels are linearly available in hidden space. It uses only high-confidence, nonblank windows to avoid training a probe on ambiguous CTC blank regions.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROBE_MIN_TOP1_PROB = 0.60
PROBE_MIN_PER_CLASS = 25
PROBE_MAX_POINTS = 50_000

for model_key, payload in exports.items():
    tokens = payload['tokens'].copy()
    hidden = payload['hidden']
    mask = (
        (tokens['top1_prob'] >= PROBE_MIN_TOP1_PROB)
        & (~tokens['top_category'].isin(['blank', 'silence', 'other']))
    )
    eligible = tokens[mask]
    if eligible.empty:
        print(model_key, 'has no eligible high-confidence nonblank windows')
        continue
    counts = eligible['top_category'].value_counts()
    keep_classes = counts[counts >= PROBE_MIN_PER_CLASS].index.tolist()
    eligible = eligible[eligible['top_category'].isin(keep_classes)]
    if eligible['top_category'].nunique() < 2:
        print(model_key, 'does not have at least two sufficiently frequent classes:', counts.to_dict())
        continue
    if len(eligible) > PROBE_MAX_POINTS:
        eligible = eligible.sample(PROBE_MAX_POINTS, random_state=7)
    x = hidden[eligible.index.to_numpy()]
    y = eligible['top_category'].to_numpy()
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.25, random_state=7, stratify=y
    )
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight='balanced', multi_class='auto'),
    )
    clf.fit(x_train, y_train)
    pred = clf.predict(x_test)
    print('\n===', model_key, '===')
    print(classification_report(y_test, pred))
    labels = sorted(set(y_test) | set(pred))
    cm = confusion_matrix(y_test, pred, labels=labels, normalize='true')
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, xticklabels=labels, yticklabels=labels, cmap='Blues', vmin=0, vmax=1, annot=True, fmt='.2f')
    plt.title(f'{model_key}: broad-category probe confusion')
    plt.xlabel('predicted')
    plt.ylabel('top category')
    plt.tight_layout()
    plt.show()

## Save Analysis Tables

Optional compact outputs for later plotting outside the notebook.

In [ ]:
ANALYSIS_DIR = EXPORT_ROOT / 'analysis_tables'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

for model_key, payload in exports.items():
    tokens = payload['tokens']
    summary = tokens.groupby('top_category').agg(
        token_count=('global_token_index', 'count'),
        mean_top1_prob=('top1_prob', 'mean'),
        mean_entropy_bits=('entropy_bits', 'mean'),
        mean_vowel_prob=('vowel_prob', 'mean'),
        mean_consonant_prob=('consonant_prob', 'mean'),
    ).reset_index()
    summary.to_csv(ANALYSIS_DIR / f'{model_key}_category_summary.csv', index=False)

    if model_key in pca_payloads:
        pca_payloads[model_key]['frame'].to_csv(ANALYSIS_DIR / f'{model_key}_pca_sample.csv', index=False)

print('Saved analysis tables to:', ANALYSIS_DIR)